# 03 - Step 2: Hybrid model
**Hybrid = all sensors x all models.** Every model gets the full multi-sensor feature set
(Sentinel-1, Sentinel-2, Landsat LST, MODIS NDVI, DEM, three seasons) and the hybrid combines them
by weighted soft voting (weights = macro-F1 on the validation blocks).

| ID | Member |
|---|---|
| H1 | Random Forest |
| H2 | SVM |
| H3 | CNN (TempCNN over seasons) |
| H4 | Transformer (attention over seasons) |
| H5 | U-Net (spatial, optional - needs the exported stack) |
| H* | **Hybrid ensemble** |

Use a GPU runtime: *Runtime -> Change runtime type -> T4 GPU*.

In [ ]:
# --- Setup (run once per Colab session) ---
!git clone -q https://github.com/veeresh0804/PrithviBodh.git 2>/dev/null || (cd PrithviBodh && git pull -q)
%cd /content/PrithviBodh
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PrithviBodh'      # Earth Engine exports land here
import os; os.makedirs(DRIVE, exist_ok=True)

import sys; sys.path.insert(0, '.')
from prithvibodh.config import load_config
cfg = load_config()
cfg['gee_project'] = 'YOUR_EE_PROJECT_ID'         # <-- your free noncommercial project
from prithvibodh import gee_data as G
G.init(cfg)

In [ ]:
import pandas as pd
from prithvibodh import experiment, evaluate
df = pd.read_csv(f'{DRIVE}/points_table.csv')
existing = pd.read_csv(f'{DRIVE}/results_existing.csv')

### Optional: U-Net (H5)
Export the stack once (takes a while, ~4 GB in Drive), then cut chips around the points.
Skip this section to run the hybrid with H1-H4 only.

In [ ]:
task = G.export_stack(cfg, cfg['train_year'])   # check progress at code.earthengine.google.com/tasks

In [ ]:
from prithvibodh.deep import UNetModel, read_chips
# Earth Engine may split a big export into several tiles: join them into one virtual file
yr = cfg['train_year']
!gdalbuildvrt -q {DRIVE}/stack_{yr}.vrt {DRIVE}/stack_{yr}*.tif
stack = f'{DRIVE}/stack_{yr}.vrt'
chip = cfg['models']['unet']['chip_size']
tr, va, te = (df[df.split == s] for s in ('train', 'val', 'test'))
Xtr, Ytr = read_chips(stack, tr, cfg, chip)
unet = UNetModel(cfg, len(cfg['classes'])).fit(Xtr, Ytr)
unet_val = unet.predict_proba_centre(read_chips(stack, va, cfg, chip)[0])
unet_test = unet.predict_proba_centre(read_chips(stack, te, cfg, chip)[0])

In [ ]:
# Run the hybrid (set the U-Net arguments to None if you skipped that section)
rows_h, out = experiment.run_hybrid(df, cfg, include_deep=True,
                                    unet_val_probs=globals().get('unet_val'),
                                    unet_test_probs=globals().get('unet_test'))
results = pd.concat([existing, evaluate.results_table(rows_h)], ignore_index=True)
results.to_csv(f'{DRIVE}/results.csv', index=False)
results

In [ ]:
# Is the hybrid really better than the best existing solution? (paired bootstrap on test points)
_, preds_e = experiment.run_existing(df, cfg)
test = df[df.split == 'test']
best = existing[existing.model.str.startswith(('E4', 'E5'))].sort_values('macro_F1').iloc[-1]['model'][:2]
lo, hi = evaluate.paired_delta_ci(out['y_test'], out['hybrid_pred'], preds_e[best][1], len(cfg['classes']))
print(f'Hybrid minus {best}: macro-F1 difference 95% CI = [{lo:.3f}, {hi:.3f}]  (above 0 = real gain)')

In [ ]:
import matplotlib.pyplot as plt
evaluate.plot_confusion(out['y_test'], out['hybrid_pred'], cfg['classes'], 'Hybrid ensemble')
plt.savefig(f'{DRIVE}/confusion_hybrid.png', dpi=150, bbox_inches='tight')
# Per-class precision, recall, F1 and IoU for the hybrid
evaluate.per_class_table(out['y_test'], out['hybrid_pred'], cfg['classes'])